# 🚀 Notebook 04b — Benchmark ANN completo sobre EMA Dialog2Flow · `version_4`

Esta notebook evalúa las representaciones EMA generadas en la notebook 03b.

Corre el benchmark ANN completo para:

- representación estática;
- acumulativa uniforme usada en el paper;
- EMA con `alpha = 0.1, ..., 0.9`.

Índices evaluados:

- `FlatL2`
- `IVF`
- `HNSW`
- `IVFPQ`

Además, persiste recuperaciones top-100 para las configuraciones representativas, que luego usa la notebook semántica.

## 1️⃣ Instalación e imports

In [ ]:
!pip install -q faiss-cpu pandas numpy tqdm scipy

In [ ]:
import json
import time
import gc
import shutil
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import faiss
from tqdm.auto import tqdm
from IPython.display import display

## 2️⃣ Montaje de Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3️⃣ Configuración

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/ANN/TF")
SOURCE_VERSION = "version_3"
VERSION = "version_4"

SOURCE_DATA_DIR = BASE_DIR / "data" / SOURCE_VERSION
SOURCE_EMBEDDINGS_DIR = SOURCE_DATA_DIR / "embeddings"

DATA_DIR = BASE_DIR / "data" / VERSION
RESULTS_DIR = BASE_DIR / "resultados" / VERSION

EMBEDDINGS_DIR = DATA_DIR / "embeddings"
SPLITS_DIR = DATA_DIR / "splits"
FAISS_INDEX_DIR = DATA_DIR / "faiss_indices"
RETRIEVAL_DIR = DATA_DIR / "retrieval_results"

BENCHMARK_RESULTS_DIR = RESULTS_DIR / "benchmark_ann"

for d in [EMBEDDINGS_DIR, SPLITS_DIR, FAISS_INDEX_DIR, RETRIEVAL_DIR, BENCHMARK_RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

EMBEDDING_LABEL = "Dialog2Flow"
SHORT_NAME = "dialog2flow"
EMBEDDING_EXPERIMENTO = "dialog2flow-joint-bert-base"

RANDOM_STATE = 42
N_QUERIES = 10_000
K_MAX = 100
NORMALIZAR_VECTORES = True
N_REPETICIONES_BUSQUEDA = 3

# Configuración ANN igual al paper
N_LIST = 4096
N_TRAIN_IVF = 100_000
NPROBE_VALUES = [1, 4, 16, 64]
NPROBE_REPRESENTATIVO = 64

HNSW_M = 32
HNSW_EF_CONSTRUCTION = 200
EF_SEARCH_VALUES = [16, 64, 128, 256]
EF_SEARCH_REPRESENTATIVO = 256

PQ_M = 32
PQ_NBITS = 8
NPROBE_VALUES_PQ = [1, 4, 16, 64]
NPROBE_PQ_REPRESENTATIVO = 64

# Persistencia
USE_SAVED_FAISS = True
SAVE_FAISS = False  # puede activarse, pero genera muchos archivos grandes
SAVE_RETRIEVAL_NPZ = True

ALPHA_VALUES = [round(x / 10, 1) for x in range(1, 10)]

print("VERSION:", VERSION)
print("ALPHA_VALUES:", ALPHA_VALUES)
print("SAVE_FAISS:", SAVE_FAISS)

## 4️⃣ Archivos y variantes

In [ ]:
def alpha_tag(alpha: float) -> str:
    return f"{alpha:.1f}".replace(".", "_")


def find_existing_file(filename: str, candidates):
    for base in candidates:
        p = base / filename
        if p.exists():
            return p
    raise FileNotFoundError(
        f"No se encontró {filename}. Busqué en: " + ", ".join(str(c) for c in candidates)
    )


static_path = find_existing_file(
    "embeddings_dialog2flow.npy",
    [EMBEDDINGS_DIR, SOURCE_EMBEDDINGS_DIR, SOURCE_DATA_DIR, BASE_DIR / "data"]
)

accumulative_path = find_existing_file(
    "accumulative_embeddings_dialog2flow.npy",
    [EMBEDDINGS_DIR, SOURCE_EMBEDDINGS_DIR, SOURCE_DATA_DIR, BASE_DIR / "data"]
)

ids_path = find_existing_file(
    "ids.npy",
    [EMBEDDINGS_DIR, SOURCE_EMBEDDINGS_DIR, SOURCE_DATA_DIR, BASE_DIR / "data"]
)

variant_paths = {
    "estatico": static_path,
    "acumulativo_uniforme": accumulative_path,
}

for alpha in ALPHA_VALUES:
    variant_paths[f"ema_alpha_{alpha_tag(alpha)}"] = EMBEDDINGS_DIR / f"ema_embeddings_dialog2flow_alpha_{alpha_tag(alpha)}.npy"

for variant, path in variant_paths.items():
    if not path.exists():
        raise FileNotFoundError(f"Falta el archivo para {variant}: {path}")

print("Variantes a evaluar:")
for v, p in variant_paths.items():
    print("-", v, "->", p)

## 5️⃣ Split fijo

In [ ]:
ids = np.load(ids_path)
n_total = len(ids)

split_index_path = SPLITS_DIR / f"index_idx_N{N_QUERIES}_seed{RANDOM_STATE}.npy"
split_query_path = SPLITS_DIR / f"query_idx_N{N_QUERIES}_seed{RANDOM_STATE}.npy"

if split_index_path.exists() and split_query_path.exists():
    index_idx = np.load(split_index_path)
    query_idx = np.load(split_query_path)
    print("Split cargado desde version_4.")
else:
    src_split_dir = SOURCE_DATA_DIR / "splits"
    src_index = src_split_dir / split_index_path.name
    src_query = src_split_dir / split_query_path.name

    if src_index.exists() and src_query.exists():
        shutil.copy(src_index, split_index_path)
        shutil.copy(src_query, split_query_path)
        index_idx = np.load(split_index_path)
        query_idx = np.load(split_query_path)
        print("Split copiado desde version_3.")
    else:
        rng = np.random.default_rng(RANDOM_STATE)
        query_idx = rng.choice(n_total, size=N_QUERIES, replace=False).astype("int64")
        mask = np.ones(n_total, dtype=bool)
        mask[query_idx] = False
        index_idx = np.where(mask)[0].astype("int64")
        np.save(split_index_path, index_idx)
        np.save(split_query_path, query_idx)
        print("Split generado y guardado.")

print("Total:", n_total)
print("Index vectors:", len(index_idx))
print("Query vectors:", len(query_idx))

## 6️⃣ Funciones auxiliares

In [ ]:
MB = 1024 ** 2


def normalize_copy(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype="float32").copy()
    if NORMALIZAR_VECTORES:
        faiss.normalize_L2(x)
    return x


def recall_at_k(indices_exactos, indices_aproximados, k: int) -> float:
    vals = []
    for exactos, aproximados in zip(indices_exactos[:, :k], indices_aproximados[:, :k]):
        vals.append(len(set(exactos).intersection(set(aproximados))) / k)
    return float(np.mean(vals))


def buscar_con_tiempo(index, queries, k):
    index.search(queries[:min(100, len(queries))], k)
    tiempos = []
    D_final, I_final = None, None
    for _ in range(N_REPETICIONES_BUSQUEDA):
        t0 = time.time()
        D, I = index.search(queries, k)
        tiempos.append(time.time() - t0)
        D_final, I_final = D, I
    return D_final, I_final, float(np.median(tiempos))


def estimate_flat_memory_mb(n, d):
    return (n * d * 4) / MB


def estimate_ivf_memory_mb(n, d, nlist):
    return ((n * d * 4) + (n * 8) + (nlist * d * 4)) / MB


def estimate_hnsw_memory_mb(n, d, M):
    return ((n * d * 4) + (n * M * 8)) / MB


def estimate_ivfpq_memory_mb(n, d, nlist, m, nbits):
    code_size_bytes = m * nbits / 8
    coarse_centroids = nlist * d * 4
    pq_codebooks = m * (2 ** nbits) * (d // m) * 4
    return ((n * code_size_bytes) + (n * 8) + coarse_centroids + pq_codebooks) / MB


def index_path(variante: str, index_type: str) -> Path:
    return FAISS_INDEX_DIR / (
        f"{SHORT_NAME}_{variante}_{index_type}"
        f"_nlist{N_LIST}_nprobe{NPROBE_REPRESENTATIVO}"
        f"_M{HNSW_M}_pq{PQ_M}x{PQ_NBITS}_seed{RANDOM_STATE}.faiss"
    )


def retrieval_path(variante: str, index_type: str, suffix: str) -> Path:
    return RETRIEVAL_DIR / (
        f"retrieval_{SHORT_NAME}_{variante}_{index_type}_{suffix}"
        f"_K{K_MAX}_N{N_QUERIES}_seed{RANDOM_STATE}.npz"
    )


def save_retrieval_npz(path: Path, distances, local_neighbors, variante: str, index_type: str):
    if not SAVE_RETRIEVAL_NPZ:
        return
    neighbor_rows = index_idx[local_neighbors]
    np.savez_compressed(
        path,
        distances=distances.astype("float32"),
        local_neighbors=local_neighbors.astype("int64"),
        neighbor_rows=neighbor_rows.astype("int64"),
        query_rows=query_idx.astype("int64"),
        index_rows=index_idx.astype("int64"),
        embedding_model=EMBEDDING_EXPERIMENTO,
        embedding_base=EMBEDDING_LABEL,
        short_name=SHORT_NAME,
        variant=variante,
        index_type=index_type,
    )


def log_row(rows, variante, index_type, params, build_time_s, search_time_s, I_rec, memory_mb, source, d):
    rows.append({
        "embedding_model": EMBEDDING_EXPERIMENTO,
        "embedding_base": EMBEDDING_LABEL,
        "short_name": SHORT_NAME,
        "variant": variante,
        "alpha": extract_alpha(variante),
        "index_type": index_type,
        "source": source,
        "params": json.dumps(params, ensure_ascii=False),
        "nlist": params.get("nlist"),
        "nprobe": params.get("nprobe"),
        "M": params.get("M"),
        "efConstruction": params.get("efConstruction"),
        "efSearch": params.get("efSearch"),
        "m": params.get("m"),
        "nbits": params.get("nbits"),
        "num_index_vectors": len(index_idx),
        "num_query_vectors": len(query_idx),
        "dimension": d,
        "build_time_s": build_time_s,
        "search_time_s": search_time_s,
        "qps": len(query_idx) / search_time_s,
        "recall@1": recall_at_k(I_exact, I_rec, 1),
        "recall@10": recall_at_k(I_exact, I_rec, 10),
        "recall@100": recall_at_k(I_exact, I_rec, 100),
        "memory_mb": memory_mb,
    })


def extract_alpha(variante: str):
    if not variante.startswith("ema_alpha_"):
        return np.nan
    return float(variante.replace("ema_alpha_", "").replace("_", "."))

## 7️⃣ Construcción/carga de índices FAISS

In [ ]:
def get_or_build_ivf(index_vectors, variante):
    path = index_path(variante, "IVF")
    if USE_SAVED_FAISS and path.exists():
        idx = faiss.read_index(str(path))
        idx.nprobe = NPROBE_REPRESENTATIVO
        return idx, 0.0, "loaded"

    d = index_vectors.shape[1]
    rng = np.random.default_rng(RANDOM_STATE)
    train_idx = rng.choice(index_vectors.shape[0], size=min(N_TRAIN_IVF, index_vectors.shape[0]), replace=False)

    quantizer = faiss.IndexFlatL2(d)
    idx = faiss.IndexIVFFlat(quantizer, d, N_LIST, faiss.METRIC_L2)

    t0 = time.time()
    idx.train(index_vectors[train_idx])
    idx.add(index_vectors)
    build_time = time.time() - t0
    idx.nprobe = NPROBE_REPRESENTATIVO

    if SAVE_FAISS:
        faiss.write_index(idx, str(path))
    return idx, build_time, "built"


def get_or_build_hnsw(index_vectors, variante):
    path = index_path(variante, "HNSW")
    if USE_SAVED_FAISS and path.exists():
        idx = faiss.read_index(str(path))
        idx.hnsw.efSearch = EF_SEARCH_REPRESENTATIVO
        return idx, 0.0, "loaded"

    d = index_vectors.shape[1]
    idx = faiss.IndexHNSWFlat(d, HNSW_M)
    idx.hnsw.efConstruction = HNSW_EF_CONSTRUCTION

    t0 = time.time()
    idx.add(index_vectors)
    build_time = time.time() - t0
    idx.hnsw.efSearch = EF_SEARCH_REPRESENTATIVO

    if SAVE_FAISS:
        faiss.write_index(idx, str(path))
    return idx, build_time, "built"


def get_or_build_ivfpq(index_vectors, variante):
    path = index_path(variante, "IVFPQ")
    if USE_SAVED_FAISS and path.exists():
        idx = faiss.read_index(str(path))
        idx.nprobe = NPROBE_PQ_REPRESENTATIVO
        return idx, 0.0, "loaded"

    d = index_vectors.shape[1]
    if d % PQ_M != 0:
        raise ValueError(f"La dimensión {d} no es divisible por PQ_M={PQ_M}")

    rng = np.random.default_rng(RANDOM_STATE)
    train_idx = rng.choice(index_vectors.shape[0], size=min(N_TRAIN_IVF, index_vectors.shape[0]), replace=False)

    quantizer = faiss.IndexFlatL2(d)
    idx = faiss.IndexIVFPQ(quantizer, d, N_LIST, PQ_M, PQ_NBITS)

    t0 = time.time()
    idx.train(index_vectors[train_idx])
    idx.add(index_vectors)
    build_time = time.time() - t0
    idx.nprobe = NPROBE_PQ_REPRESENTATIVO

    if SAVE_FAISS:
        faiss.write_index(idx, str(path))
    return idx, build_time, "built"

## 8️⃣ Benchmark ANN completo

In [ ]:
all_results = []

for variante, emb_path in variant_paths.items():
    print("=" * 100)
    print("Procesando:", EMBEDDING_LABEL, "|", variante)
    print("Archivo:", emb_path)

    embeddings = np.load(emb_path, mmap_mode="r")

    index_vectors = normalize_copy(embeddings[index_idx])
    query_vectors = normalize_copy(embeddings[query_idx])

    n_index, d = index_vectors.shape
    print("index_vectors:", index_vectors.shape)
    print("query_vectors:", query_vectors.shape)

    # --------------------------------------------------------
    # FlatL2 exacto
    # --------------------------------------------------------
    t0 = time.time()
    index_flat = faiss.IndexFlatL2(d)
    index_flat.add(index_vectors)
    build_time_flat = time.time() - t0

    D_exact, I_exact, search_time_flat = buscar_con_tiempo(index_flat, query_vectors, K_MAX)
    save_retrieval_npz(retrieval_path(variante, "FlatL2", "exact"), D_exact, I_exact, variante, "FlatL2")

    all_results.append({
        "embedding_model": EMBEDDING_EXPERIMENTO,
        "embedding_base": EMBEDDING_LABEL,
        "short_name": SHORT_NAME,
        "variant": variante,
        "alpha": extract_alpha(variante),
        "index_type": "FlatL2",
        "source": "built",
        "params": json.dumps({}),
        "nlist": None,
        "nprobe": None,
        "M": None,
        "efConstruction": None,
        "efSearch": None,
        "m": None,
        "nbits": None,
        "num_index_vectors": len(index_idx),
        "num_query_vectors": len(query_idx),
        "dimension": d,
        "build_time_s": build_time_flat,
        "search_time_s": search_time_flat,
        "qps": len(query_idx) / search_time_flat,
        "recall@1": 1.0,
        "recall@10": 1.0,
        "recall@100": 1.0,
        "memory_mb": estimate_flat_memory_mb(n_index, d),
    })
    print("FlatL2 OK", "QPS:", len(query_idx) / search_time_flat)

    del index_flat, D_exact
    gc.collect()

    # --------------------------------------------------------
    # IVF
    # --------------------------------------------------------
    index_ivf, build_time_ivf, source_ivf = get_or_build_ivf(index_vectors, variante)
    mem_ivf = estimate_ivf_memory_mb(n_index, d, N_LIST)

    for nprobe in NPROBE_VALUES:
        index_ivf.nprobe = nprobe
        D, I, st = buscar_con_tiempo(index_ivf, query_vectors, K_MAX)
        log_row(all_results, variante, "IVF", {"nlist": N_LIST, "nprobe": nprobe}, build_time_ivf, st, I, mem_ivf, source_ivf, d)
        if nprobe == NPROBE_REPRESENTATIVO:
            save_retrieval_npz(retrieval_path(variante, "IVF", f"nprobe{nprobe}"), D, I, variante, "IVF")
        print("IVF", nprobe, "QPS:", len(query_idx) / st)

    del index_ivf, D, I
    gc.collect()

    # --------------------------------------------------------
    # HNSW
    # --------------------------------------------------------
    index_hnsw, build_time_hnsw, source_hnsw = get_or_build_hnsw(index_vectors, variante)
    mem_hnsw = estimate_hnsw_memory_mb(n_index, d, HNSW_M)

    for ef in EF_SEARCH_VALUES:
        index_hnsw.hnsw.efSearch = ef
        D, I, st = buscar_con_tiempo(index_hnsw, query_vectors, K_MAX)
        log_row(
            all_results,
            variante,
            "HNSW",
            {"M": HNSW_M, "efConstruction": HNSW_EF_CONSTRUCTION, "efSearch": ef},
            build_time_hnsw,
            st,
            I,
            mem_hnsw,
            source_hnsw,
            d,
        )
        if ef == EF_SEARCH_REPRESENTATIVO:
            save_retrieval_npz(retrieval_path(variante, "HNSW", f"efSearch{ef}"), D, I, variante, "HNSW")
        print("HNSW", ef, "QPS:", len(query_idx) / st)

    del index_hnsw, D, I
    gc.collect()

    # --------------------------------------------------------
    # IVFPQ
    # --------------------------------------------------------
    index_ivfpq, build_time_ivfpq, source_ivfpq = get_or_build_ivfpq(index_vectors, variante)
    mem_ivfpq = estimate_ivfpq_memory_mb(n_index, d, N_LIST, PQ_M, PQ_NBITS)

    for nprobe in NPROBE_VALUES_PQ:
        index_ivfpq.nprobe = nprobe
        D, I, st = buscar_con_tiempo(index_ivfpq, query_vectors, K_MAX)
        log_row(
            all_results,
            variante,
            "IVFPQ",
            {"nlist": N_LIST, "m": PQ_M, "nbits": PQ_NBITS, "nprobe": nprobe},
            build_time_ivfpq,
            st,
            I,
            mem_ivfpq,
            source_ivfpq,
            d,
        )
        if nprobe == NPROBE_PQ_REPRESENTATIVO:
            save_retrieval_npz(retrieval_path(variante, "IVFPQ", f"nprobe{nprobe}"), D, I, variante, "IVFPQ")
        print("IVFPQ", nprobe, "QPS:", len(query_idx) / st)

    del index_ivfpq, D, I, embeddings, index_vectors, query_vectors, I_exact
    gc.collect()

print("Benchmark finalizado.")

## 9️⃣ Resultados y guardado

In [ ]:
df_results = pd.DataFrame(all_results)
display(df_results)

timestamp = datetime.now(ZoneInfo("America/Argentina/Buenos_Aires")).strftime("%Y%m%d_%H%M%S_ARG")

benchmark_path = BENCHMARK_RESULTS_DIR / f"resultados_ann_ema_dialog2flow_{timestamp}.csv"
benchmark_latest_path = BENCHMARK_RESULTS_DIR / "resultados_ann_ema_dialog2flow_latest.csv"
metadata_path = BENCHMARK_RESULTS_DIR / f"metadata_ann_ema_dialog2flow_{timestamp}.json"

df_results.to_csv(benchmark_path, index=False)
df_results.to_csv(benchmark_latest_path, index=False)

metadata = {
    "version": VERSION,
    "source_version": SOURCE_VERSION,
    "embedding_experimento": EMBEDDING_EXPERIMENTO,
    "embedding_label": EMBEDDING_LABEL,
    "short_name": SHORT_NAME,
    "random_state": RANDOM_STATE,
    "n_queries": N_QUERIES,
    "k_max": K_MAX,
    "alpha_values": ALPHA_VALUES,
    "variants": list(variant_paths.keys()),
    "n_list": N_LIST,
    "nprobe_values": NPROBE_VALUES,
    "nprobe_representativo": NPROBE_REPRESENTATIVO,
    "hnsw_m": HNSW_M,
    "hnsw_ef_construction": HNSW_EF_CONSTRUCTION,
    "ef_search_values": EF_SEARCH_VALUES,
    "ef_search_representativo": EF_SEARCH_REPRESENTATIVO,
    "pq_m": PQ_M,
    "pq_nbits": PQ_NBITS,
    "nprobe_values_pq": NPROBE_VALUES_PQ,
    "nprobe_pq_representativo": NPROBE_PQ_REPRESENTATIVO,
    "save_faiss": SAVE_FAISS,
    "save_retrieval_npz": SAVE_RETRIEVAL_NPZ,
    "timestamp": timestamp,
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Benchmark guardado en:", benchmark_path)
print("Latest guardado en:", benchmark_latest_path)
print("Metadata guardada en:", metadata_path)

## 🔟 Vista compacta

In [ ]:
cols = [
    "variant", "alpha", "index_type", "nprobe", "efSearch",
    "qps", "recall@1", "recall@10", "recall@100", "memory_mb"
]

summary_view = df_results[cols].sort_values(
    ["variant", "index_type", "nprobe", "efSearch"],
    ascending=[True, True, True, True]
)

display(summary_view)